# 고속도로 휴게소 기준정보 현황

In [1]:
import os
import time
import math
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
DATA_EX_KEY = os.getenv("DATA_EX_KEY")
if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")

base_url = "https://data.ex.co.kr/openapi/locationinfo/locationinfoRest"

# 1) 먼저 첫 페이지로 총 개수 확인
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": 100, "pageNo": 1}
response = requests.get(base_url, params=params, timeout=10)
response.raise_for_status()
data_json = response.json()

# 인증/오류 체크
if data_json.get("code") == "ERROR":
    raise RuntimeError(f"api error: {data_json.get('message')}")

total_count = data_json.get("count", 0)
if not total_count:
    print("결과값이 0이거나 응답에 count가 없습니다. 전체 응답:", data_json)
    df_all = pd.DataFrame()
else:
    per_page = params["numOfRows"]
    total_pages = math.ceil(total_count / per_page)
    print(f"총 항목: {total_count}, 페이지 수: {total_pages}, 페이지당: {per_page}")

    frames = []
    for page in range(1, total_pages + 1):
        params["pageNo"] = page
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            json = response.json()
        except requests.RequestException as e:
            print(f"페이지 {page} 요청 실패: {e}")
            break

        # 인증/에러 메시지 재확인
        if json.get("code") == "ERROR":
            raise RuntimeError(f"api error on page {page}: {j.get('message')}")

        page_list = json.get("list")
        if page_list:
            frames.append(pd.json_normalize(page_list))
        else:
            print(f"페이지 {page}의 'list'가 비어있음 또는 null")

        # 서버 과부하 방지용 짧은 대기
        time.sleep(0.2)

    if frames:
        df_all = pd.concat(frames, ignore_index=True)
    else:
        df_all = pd.DataFrame()

print("최종 DataFrame shape:", df_all.shape)
print(df_all.head())

총 항목: 203, 페이지 수: 3, 페이지당: 100
최종 DataFrame shape: (203, 10)
      unitName pageNo numOfRows unitCode routeName routeNo      xValue  \
0  서울만남(부산)휴게소   None      None      001       경부선    0010  127.042514   
1    죽전(서울)휴게소   None      None      002       경부선    0010  127.104397   
2    기흥(부산)휴게소   None      None      003       경부선    0010  127.105190   
3    안성(서울)휴게소   None      None      004       경부선    0010  127.131800   
4    안성(부산)휴게소   None      None      005       경부선    0010  127.145006   

      yValue stdRestCd serviceAreaCode  
0  37.459939    000001          A00001  
1  37.332583    000003          A00002  
2  37.235090    000005          A00003  
3  37.076443    000007          A00004  
4  37.013628    000009          A00005  


# geopandas를 이용해서 좌표를 권역별로 묶어주기

In [2]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# ---------- 사용자 설정 (파일명/변수 확인) ----------
SIGUNGU_GEOJSON = "korea_sigungu.geojson"   # 시군구 GeoJSON 경로
SIDO_GEOJSON = "korea_sido.geojson"         # 시도 GeoJSON 경로
DF = df_all                                 # 이미 로드된 DataFrame (unitName, xValue, yValue)

# ---------- 1) 데이터 준비 ----------
df = DF.copy().dropna(subset=["xValue","yValue"]).reset_index(drop=True)
df["xValue"] = df["xValue"].astype(float)
df["yValue"] = df["yValue"].astype(float)
gdf_points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["xValue"], df["yValue"]), crs="EPSG:4326")

# ---------- 2) 경계 파일 로드 및 CRS 통일 ----------
sigungu = gpd.read_file(SIGUNGU_GEOJSON, encoding="utf-8").to_crs("EPSG:4326")
sido = gpd.read_file(SIDO_GEOJSON, encoding="utf-8").to_crs("EPSG:4326")

# 충돌 가능 컬럼 제거(안전)
for g in [gdf_points, sigungu, sido]:
    if "index_right" in g.columns:
        g.drop(columns=["index_right"], inplace=True)

# ---------- 3) 포인트 -> 시군구 공간조인 (기본) ----------
gdf_points = gdf_points.reset_index(drop=True)
sigungu = sigungu.reset_index(drop=True)
joined = gpd.sjoin(gdf_points, sigungu, how="left", predicate="within")

# ---------- 4) 시도명 채우기 (시군구 properties에 시도명 컬럼 'CTP_KOR_NM' 사용) ----------
# sigungu의 properties에서 시도명 컬럼이 'CTP_KOR_NM'로 확인됨
joined["province_name"] = joined.get("CTP_KOR_NM")

# ---------- 5) 시군구에서 못 찾은 경우 시도 레이어로 재시도 ----------
missing_idx = joined[joined["province_name"].isna()].index
if len(missing_idx) > 0:
    missing_points = joined.loc[missing_idx].drop(columns=["index_right"], errors="ignore")
    rejoined = gpd.sjoin(missing_points, sido, how="left", predicate="within")
    if "CTP_KOR_NM" in rejoined.columns:
        joined.loc[missing_idx, "province_name"] = rejoined["CTP_KOR_NM"].values
    else:
        # fallback: 시도 GeoDataFrame의 문자열 컬럼 중 첫 번째 사용
        str_cols = [c for c in sido.columns if sido[c].dtype == "object"]
        if str_cols:
            joined.loc[missing_idx, "province_name"] = rejoined[str_cols[0]].values

# ---------- 6) 버퍼 재시도 (경계 근처 보정, 미터 단위) ----------
still_unassigned_idx = joined[joined["province_name"].isna()].index
if len(still_unassigned_idx) > 0:
    still = joined.loc[still_unassigned_idx].copy()
    still = still.drop(columns=["index_right"], errors="ignore")
    # 투영(미터 단위) — 한국 권장: EPSG:5179
    PROJECTED_CRS = "EPSG:5179"
    still_proj = still.to_crs(PROJECTED_CRS)
    sigungu_proj = sigungu.to_crs(PROJECTED_CRS)
    # 버퍼 크기(미터) — 필요시 조정: 50, 100, 200 등
    buffer_m = 200
    still_proj["geometry"] = still_proj.geometry.buffer(buffer_m)
    # 재조인
    if "index_right" in sigungu_proj.columns:
        sigungu_proj = sigungu_proj.drop(columns=["index_right"])
    rejoin_buf = gpd.sjoin(still_proj, sigungu_proj, how="left", predicate="intersects")
    # 안전하게 province 채우기
    preferred = ["CTP_KOR_NM", "CTPRVN_CD", "CTP_ENG_NM"]
    for idx, row in rejoin_buf.iterrows():
        # left index corresponds to joined index
        left_idx = idx
        prov = None
        for c in preferred:
            if c in row.index and pd.notna(row[c]):
                prov = row[c]; break
        if prov is None:
            # 문자열 컬럼 후보에서 첫값 사용
            str_cols = [c for c in rejoin_buf.columns if rejoin_buf[c].dtype == "object"]
            for c in str_cols:
                if pd.notna(row[c]):
                    prov = row[c]; break
        if prov is not None and left_idx in joined.index:
            joined.at[left_idx, "province_name"] = prov

# ---------- 7) province_name -> 권역 코드 매핑 ----------
province_to_region_geo = {
    "서울특별시": "SEOUL_GYEONGGI", "경기도": "SEOUL_GYEONGGI", "인천광역시": "SUDOGWON",
    "강원도": "GANGWON",
    "대전광역시": "DAEJEON_CHUNGNAM", "충청남도": "DAEJEON_CHUNGNAM",
    "전라북도": "JEONBUK",
    "광주광역시": "GWANGJU_JEONNAM", "전라남도": "GWANGJU_JEONNAM",
    "대구광역시": "DAEGU_GYEONGBUK", "경상북도": "DAEGU_GYEONGBUK",
    "부산광역시": "BUSAN_GYEONGNAM", "경상남도": "BUSAN_GYEONGNAM",
    "충청북도": "CHUNGBUK"
}
joined["region"] = joined["province_name"].map(province_to_region_geo)

# ---------- 8) 결과 요약 및 저장 ----------
assigned = joined[~joined["region"].isna()]
unassigned_final = joined[joined["region"].isna()]

print("총 레코드:", len(joined))
print("권역 할당된 수:", len(assigned))
print("최종 미할당 수:", len(unassigned_final))

# 권역별 리스트 (딕셔너리)
groups = {region: g["unitName"].tolist() for region, g in assigned.groupby("region")}

# 권역별 CSV 저장 (원하면)
for region, items in groups.items():
    pd.Series(items, name="unitName").to_csv(f"restareas_{region}.csv", index=False, encoding="utf-8-sig")

# 미할당 저장(검토용)
unassigned_final.drop(columns=["geometry"], errors="ignore").to_csv("restareas_unassigned_geo.csv", index=False, encoding="utf-8-sig")

총 레코드: 203
권역 할당된 수: 117
최종 미할당 수: 86


In [ ]:
import re
import pandas as pd

# 기존 매핑(필요하면 확장)
province_to_headquarter = {
    "서울특별시":"서울경기본부","경기도":"서울경기본부","인천광역시":"수도권본부",
    "강원도":"강원본부",
    "대전광역시":"대전충남본부","충청남도":"대전충남본부",
    "전라북도":"전북본부",
    "광주광역시":"광주전남본부","전라남도":"광주전남본부",
    "대구광역시":"대구경북본부","경상북도":"대구경북본부",
    "부산광역시":"부산경남본부","경상남도":"부산경남본부",
    "충청북도":"충북본부"
}
headquarter_to_region = {
    '수도권본부': 'SUDOGWON','서울경기본부': 'SEOUL_GYEONGGI','강원본부': 'GANGWON',
    '대전충남본부': 'DAEJEON_CHUNGNAM','전북본부': 'JEONBUK','광주전남본부': 'GWANGJU_JEONNAM',
    '대구경북본부': 'DAEGU_GYEONGBUK','부산경남본부': 'BUSAN_GYEONGNAM','충북본부': 'CHUNGBUK'
}

# 보정용 보조표: 축약형/다른 표기 -> 표준 시도명
normalize_map = {
    "전북":"전라북도","전남":"전라남도","충북":"충청북도","충남":"충청남도",
    "경북":"경상북도","경남":"경상남도","경기":"경기도","서울":"서울특별시",
    "부산":"부산광역시","대구":"대구광역시","광주":"광주광역시","대전":"대전광역시",
    "강원":"강원도","인천":"인천광역시","제주":"제주특별자치도"
}

def normalize_prov_text(s):
    if pd.isna(s):
        return None
    s0 = str(s).strip()
    # 괄호나 추가 표기 제거 (예: "죽전(서울)휴게소" -> "서울")
    # 괄호 안의 텍스트 우선 추출
    m = re.search(r"\(([^)]+)\)", s0)
    if m:
        inner = m.group(1)
        # inner가 시도/광역시/도 표기면 사용
        for key in normalize_map:
            if key in inner:
                return normalize_map[key]
        # inner가 이미 '서울' 등이라면 확장 표기 적용
        if inner in normalize_map:
            return normalize_map[inner]
        # inner가 '서울'처럼 축약이면 확장
        if inner in ["서울","부산","대구","광주","대전","인천","제주"]:
            return normalize_map.get(inner)
    # 괄호 없거나 실패 시, 문자열에서 시도명 키워드 찾기
    for k in list(province_to_headquarter.keys()):
        if k in s0:
            return k
    # 축약형 매핑
    for short, full in normalize_map.items():
        if short in s0:
            return full
    # 마지막으로 숫자/특수문자 제거 후 단어 매칭
    s_clean = re.sub(r"[^가-힣 ]", " ", s0)
    for short, full in normalize_map.items():
        if short in s_clean:
            return full
    return None

# 적용: province_name을 시도 표준명으로 변환한 컬럼 생성
joined["province_standard"] = joined["province_name"].apply(normalize_prov_text)

# map to headquarter/region
joined["headquarter"] = joined["province_standard"].map(province_to_headquarter)
joined["region"] = joined["headquarter"].map(headquarter_to_region)

# 결과 요약
before = len(joined[~joined["region"].isna()])
after = len(joined[~joined["region"].isna()])
print("권역 할당 수 (보정 적용 후):", after)
print("province_standard 고유값 분포:")
print(joined["province_standard"].value_counts(dropna=False).to_string())
display(joined[joined["region"].isna()][["unitName","xValue","yValue","province_name","province_standard"]].head(50))

In [ ]:
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 전제: sigungu_proj (시군구, EPSG:4326) 에 'sido_name' 컬럼이 있어야 함
#        joined (현재 결과) 존재

# 1) city -> sido 매핑 테이블 생성 (sigungu의 SIG_KOR_NM -> sido_name)
city_to_sido = {}
if "SIG_KOR_NM" in sigungu.columns and "sido_name" in sigungu.columns:
    for i, r in sigungu.iterrows():
        city = r["SIG_KOR_NM"]
        sido = r["sido_name"]
        if pd.notna(city) and pd.notna(sido):
            city_to_sido[city.strip()] = sido.strip()

# 2) 유틸: unitName에서 괄호 안 텍스트 추출, 또는 휴게소명 토큰 추출
def extract_city_tokens(name):
    if pd.isna(name):
        return []
    s = str(name)
    tokens = []
    # 괄호 안 우선
    m = re.findall(r"\(([^)]+)\)", s)
    for x in m:
        tokens.append(x.strip())
    # 괄호 없거나 추가로, 괄호 앞/뒤의 단어들(예: '진영(순천)휴게소' -> '진영','순천')
    s_clean = re.sub(r"[^가-힣 ]", " ", s)
    parts = s_clean.split()
    for p in parts:
        if p and len(p) <= 4:  # 짧은 지명 후보
            tokens.append(p.strip())
    # 중복 제거, 우선순위 유지
    seen = set(); out=[]
    for t in tokens:
        if t not in seen:
            seen.add(t); out.append(t)
    return out

# 3) 보정 함수: 여러 전략으로 province_standard 채우기
def infer_sido_for_row(row):
    # 이미 표준화된 값이 있으면 그대로
    if pd.notna(row.get("province_standard")):
        return row["province_standard"]
    # 3a: unitName에서 토큰 추출 -> city_to_sido 매핑
    tokens = extract_city_tokens(row.get("unitName"))
    for tok in tokens:
        # 직접 city 이름 일치
        if tok in city_to_sido:
            return city_to_sido[tok]
        # 축약형 매핑(예: '순천' -> '전라남도')
        # city_to_sido 키 중에 tok가 포함된 항목 찾기 (부분매칭)
        for city_name, sido_name in city_to_sido.items():
            if tok in city_name:
                return sido_name
    # 3b: 현재 province_name이 시군구명(예: '진영(순천)휴게소' 또는 '진영')이면 시군구명으로 매핑 시도
    prov = row.get("province_name")
    if pd.notna(prov):
        p = str(prov).strip()
        # 직접 일치하는 시군구명 찾기
        if p in city_to_sido:
            return city_to_sido[p]
        # 괄호 안 추출
        m = re.findall(r"\(([^)]+)\)", p)
        for tok in m:
            if tok in city_to_sido:
                return city_to_sido[tok]
        # 부분 포함 매칭
        for city_name, sido_name in city_to_sido.items():
            if city_name in p or p in city_name:
                return sido_name
    # 3c: 공간적으로 근접한 시군구 찾기 (투영 후 500m)
    try:
        pt = Point(row["xValue"], row["yValue"])
        # 투영
        sig_proj = sigungu.to_crs("EPSG:5179")
        pt_g = gpd.GeoSeries([pt], crs="EPSG:4326").to_crs("EPSG:5179").iloc[0]
        # 버퍼 500m
        buf = pt_g.buffer(500)
        # intersects 검사
        candidates = sig_proj[sig_proj.intersects(buf)]
        if not candidates.empty:
            # 가장 면적이 큰(혹은 첫번째) 후보의 sido_name 반환
            cand = candidates.iloc[0]
            if pd.notna(cand.get("sido_name")):
                return cand["sido_name"]
    except Exception:
        pass
    return None

# 4) 적용: 미할당(또는 province_standard None) 행에 대해 infer 시도
joined["province_standard"] = joined.get("province_standard")  # 기존 컬럼 유지
mask = joined["province_standard"].isna()
for idx, row in joined[mask].iterrows():
    inferred = infer_sido_for_row(row)
    if inferred:
        joined.at[idx, "province_standard"] = inferred

# 5) 매핑 갱신: province_standard -> headquarter -> region
province_to_headquarter = {
    "서울특별시":"서울경기본부","경기도":"서울경기본부","인천광역시":"수도권본부",
    "강원도":"강원본부",
    "대전광역시":"대전충남본부","충청남도":"대전충남본부",
    "전라북도":"전북본부",
    "광주광역시":"광주전남본부","전라남도":"광주전남본부",
    "대구광역시":"대구경북본부","경상북도":"대구경북본부",
    "부산광역시":"부산경남본부","경상남도":"부산경남본부",
    "충청북도":"충북본부","제주특별자치도":"JEJU"  # 필요시 확장
}
headquarter_to_region = {
    '수도권본부': 'SUDOGWON','서울경기본부': 'SEOUL_GYEONGGI','강원본부': 'GANGWON',
    '대전충남본부': 'DAEJEON_CHUNGNAM','전북본부': 'JEONBUK','광주전남본부': 'GWANGJU_JEONNAM',
    '대구경북본부': 'DAEGU_GYEONGBUK','부산경남본부': 'BUSAN_GYEONGNAM','충북본부': 'CHUNGBUK'
}

joined["headquarter"] = joined["province_standard"].map(province_to_headquarter)
joined["region"] = joined["headquarter"].map(headquarter_to_region)

# 6) 결과 요약 출력
total = len(joined)
assigned = joined[~joined["region"].isna()].shape[0]
unassigned = joined[joined["region"].isna()].shape[0]
print("총:", total, "할당:", assigned, "미할당:", unassigned)
display(joined[joined["region"].isna()][["unitName","xValue","yValue","province_name","province_standard"]].head(50))

In [ ]:
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 전제: joined, sigungu, sido가 이미 로드되어 있고
# sigungu에는 'SIG_KOR_NM'과 'sido_name'이 있어야 함 (이전 단계에서 생성됨)

# 0) 현재 미할당 리스트
mask_none = joined["province_standard"].isna()
print("초기 province_standard None 개수:", mask_none.sum())

# 1) 보정용 사전 확장: 축약형/도시->시도 매핑 (필요시 더 추가)
short_to_sido = {
    "전북":"전라북도","전남":"전라남도","충북":"충청북도","충남":"충청남도",
    "경북":"경상북도","경남":"경상남도","경기":"경기도","서울":"서울특별시",
    "부산":"부산광역시","대구":"대구광역시","광주":"광주광역시","대전":"대전광역시",
    "강원":"강원도","인천":"인천광역시","제주":"제주특별자치도","울산":"울산광역시",
    "전주":"전라북도","순천":"전라남도","목포":"전라남도","무안":"전라남도",
    "창원":"경상남도","통영":"경상남도","영암":"전라남도","익산":"전라북도",
    "완주":"전라북도","논산":"충청남도","공주":"충청남도","당진":"충청남도",
    "서천":"충청남도","부안":"전라북도","고창":"전라북도","정읍":"전라북도",
    "함평":"전라남도","영광":"전라남도","보성":"전라남도","해남":"전라남도",
    "영암":"전라남도","군산":"전라북도","익산":"전라북도"
}

# 2) 시군구명 -> 시도명 매핑 테이블 (sigungu의 SIG_KOR_NM -> sido_name)
city_to_sido = {}
if "SIG_KOR_NM" in sigungu.columns and "sido_name" in sigungu.columns:
    for i, r in sigungu.iterrows():
        city = str(r["SIG_KOR_NM"]).strip()
        sido = r["sido_name"]
        if pd.notna(city) and pd.notna(sido):
            city_to_sido[city] = sido

# 3) 토큰 추출 함수 (unitName 또는 province_name에서)
def extract_tokens(s):
    if pd.isna(s): return []
    s = str(s)
    tokens = []
    # 괄호 안 우선
    tokens += re.findall(r"\(([^)]+)\)", s)
    # 괄호 없을 때 단어 토큰 (짧은 지명 후보)
    s_clean = re.sub(r"[^가-힣 ]", " ", s)
    parts = [p.strip() for p in s_clean.split() if p.strip()]
    tokens += [p for p in parts if len(p) <= 6]  # 길이 제한
    # 중복 제거
    out = []
    for t in tokens:
        if t not in out:
            out.append(t)
    return out

# 4) 보정 함수: 여러 전략을 순차 적용
def infer_sido(row):
    # 이미 채워진 경우
    if pd.notna(row.get("province_standard")):
        return row["province_standard"]
    # A: unitName 괄호/토큰 -> short_to_sido 우선
    for tok in extract_tokens(row.get("unitName")):
        if tok in short_to_sido:
            return short_to_sido[tok]
    # B: province_name(시군구명 형태)에서 괄호/토큰 추출 -> city_to_sido 매핑
    for tok in extract_tokens(row.get("province_name")):
        if tok in city_to_sido:
            return city_to_sido[tok]
        if tok in short_to_sido:
            return short_to_sido[tok]
    # C: province_name 전체가 시군구명일 경우 직접 매핑
    prov = row.get("province_name")
    if pd.notna(prov):
        p = str(prov).strip()
        if p in city_to_sido:
            return city_to_sido[p]
        # 부분 포함 매칭 (city_name contains token or vice versa)
        for city_name, sido_name in city_to_sido.items():
            if city_name in p or p in city_name:
                return sido_name
    # D: 공간 근접 탐색 (투영 후 500m 이내 시군구 후보)
    try:
        pt = Point(row["xValue"], row["yValue"])
        sig_proj = sigungu.to_crs("EPSG:5179")
        pt_proj = gpd.GeoSeries([pt], crs="EPSG:4326").to_crs("EPSG:5179").iloc[0]
        buf = pt_proj.buffer(500)
        cand = sig_proj[sig_proj.intersects(buf)]
        if not cand.empty:
            # 후보 중 sido_name이 있는 첫 항목 반환
            for _, c in cand.iterrows():
                if pd.notna(c.get("sido_name")):
                    return c["sido_name"]
    except Exception:
        pass
    # E: unitName에 포함된 도시 키워드로 short_to_sido 매핑(부분매칭)
    for tok in extract_tokens(row.get("unitName")):
        for short, full in short_to_sido.items():
            if tok == short or tok in short or short in tok:
                return full
    return None

# 5) 적용: province_standard가 None인 행에 대해 infer 시도
mask = joined["province_standard"].isna()
for idx, row in joined[mask].iterrows():
    inferred = infer_sido(row)
    if inferred:
        joined.at[idx, "province_standard"] = inferred

# 6) 매핑 갱신
province_to_headquarter = {
    "서울특별시":"서울경기본부","경기도":"서울경기본부","인천광역시":"수도권본부",
    "강원도":"강원본부",
    "대전광역시":"대전충남본부","충청남도":"대전충남본부",
    "전라북도":"전북본부",
    "광주광역시":"광주전남본부","전라남도":"광주전남본부",
    "대구광역시":"대구경북본부","경상북도":"대구경북본부",
    "부산광역시":"부산경남본부","경상남도":"부산경남본부",
    "충청북도":"충북본부","제주특별자치도":"JEJU"
}
headquarter_to_region = {
    '수도권본부': 'SUDOGWON','서울경기본부': 'SEOUL_GYEONGGI','강원본부': 'GANGWON',
    '대전충남본부': 'DAEJEON_CHUNGNAM','전북본부': 'JEONBUK','광주전남본부': 'GWANGJU_GEONNAM',
    '대구경북본부': 'DAEGU_GYEONGBUK','부산경남본부': 'BUSAN_GYEONGNAM','충북본부': 'CHUNGBUK'
}
joined["headquarter"] = joined["province_standard"].map(province_to_headquarter)
joined["region"] = joined["headquarter"].map(headquarter_to_region)

# 7) 결과 요약
print("보정 후 province_standard None 개수:", joined["province_standard"].isna().sum())
print("보정 후 권역 할당 수:", joined[~joined["region"].isna()].shape[0])
print("보정 후 미할당 수:", joined[joined["region"].isna()].shape[0])
display(joined[joined["region"].isna()][["unitName","xValue","yValue","province_name","province_standard"]].head(50))

In [6]:
# 1) 수동 매핑 딕셔너리
manual_map = {
    "행담도휴게소": "충청남도",
    "대산상쉼터": "충청남도",
    "곡성(천안)휴게소": "충청남도",
    "백양사(천안)휴게소": "충청남도",
    "주암(천안)휴게소": "충청남도",
    "이서(천안)휴게소": "충청남도",
    "관촌(광양)휴게소": "전라남도",
    "칠서(양평)휴게소": "경기도",
    "홍천(양양)휴게소": "강원도"
}

# 2) province_standard가 비어있거나 None인 행에 대해 unitName 기준으로 수동 매핑 적용
for idx, row in joined[joined["province_standard"].isna()].iterrows():
    name = str(row["unitName"])
    if name in manual_map:
        joined.at[idx, "province_standard"] = manual_map[name]

# 3) 매핑 테이블(기존)로 headquarter, region 갱신
province_to_headquarter = {
    "서울특별시":"서울경기본부","경기도":"서울경기본부","인천광역시":"수도권본부",
    "강원도":"강원본부",
    "대전광역시":"대전충남본부","충청남도":"대전충남본부",
    "전라북도":"전북본부",
    "광주광역시":"광주전남본부","전라남도":"광주전남본부",
    "대구광역시":"대구경북본부","경상북도":"대구경북본부",
    "부산광역시":"부산경남본부","경상남도":"부산경남본부",
    "충청북도":"충북본부","제주특별자치도":"JEJU"
}
headquarter_to_region = {
    '수도권본부': 'SUDOGWON','서울경기본부': 'SEOUL_GYEONGGI','강원본부': 'GANGWON',
    '대전충남본부': 'DAEJEON_CHUNGNAM','전북본부': 'JEONBUK','광주전남본부': 'GWANGJU_JEONNAM',
    '대구경북본부': 'DAEGU_GYEONGBUK','부산경남본부': 'BUSAN_GYEONGNAM','충북본부': 'CHUNGBUK'
}

joined["headquarter"] = joined["province_standard"].map(province_to_headquarter)
joined["region"] = joined["headquarter"].map(headquarter_to_region)

# 4) 결과 요약 출력
total = len(joined)
assigned = joined[~joined["region"].isna()].shape[0]
unassigned = joined[joined["region"].isna()].shape[0]
print("총:", total, "할당:", assigned, "미할당:", unassigned)
display(joined[joined["region"].isna()][["unitName","xValue","yValue","province_name","province_standard"]])

총: 203 할당: 203 미할당: 0


,unitName,xValue,yValue,province_name,province_standard


In [9]:
# 전체 joined 테이블을 저장 (필요 시 특정 컬럼만 선택)
joined.to_csv("joined_with_region.csv", index=False, encoding="utf-8-sig")

print("전체 joined를 joined_with_region.csv로 저장했습니다. 총 행 수:", len(joined))

전체 joined를 joined_with_region.csv로 저장했습니다. 총 행 수: 203


In [8]:
import os
from datetime import datetime
import tempfile

# 출력 폴더(없으면 생성)
out_dir = "output_assigned"
os.makedirs(out_dir, exist_ok=True)

# 타임스탬프
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1) 전체 할당된 항목만 추출
assigned_df = joined[~joined["region"].isna()].copy()

# 2) 전체 파일명 및 저장 시도
out_all = os.path.join(out_dir, f"restareas_assigned_all_{ts}.csv")
try:
    assigned_df.drop(columns=["geometry"], errors="ignore").to_csv(out_all, index=False, encoding="utf-8-sig")
    print("전체 할당 저장 완료:", out_all)
except PermissionError as e:
    print("PermissionError 발생, 임시파일로 저장합니다:", e)
    tmp = tempfile.NamedTemporaryFile(prefix="restareas_assigned_all_", suffix=".csv", delete=False, dir=out_dir)
    tmp_name = tmp.name
    tmp.close()
    assigned_df.drop(columns=["geometry"], errors="ignore").to_csv(tmp_name, index=False, encoding="utf-8-sig")
    print("임시 파일 저장:", tmp_name)
except Exception as e:
    print("전체 저장 실패:", e)

# # 3) 권역별로 분리 저장 (region 값이 유효한 경우만)
# regions = assigned_df["region"].dropna().unique().tolist()
# region_files = {}
# for r in regions:
#     safe_r = str(r).replace(" ", "_")
#     fname = os.path.join(out_dir, f"restareas_assigned_{safe_r}_{ts}.csv")
#     try:
#         assigned_df[assigned_df["region"] == r].drop(columns=["geometry"], errors="ignore").to_csv(fname, index=False, encoding="utf-8-sig")
#         region_files[r] = fname
#     except Exception as e:
#         # 실패 시 임시파일로 저장
#         tmp = tempfile.NamedTemporaryFile(prefix=f"restareas_assigned_{safe_r}_", suffix=".csv", delete=False, dir=out_dir)
#         tmp_name = tmp.name
#         tmp.close()
#         assigned_df[assigned_df["region"] == r].drop(columns=["geometry"], errors="ignore").to_csv(tmp_name, index=False, encoding="utf-8-sig")
#         region_files[r] = tmp_name

# # 4) 저장 요약 출력
# print("권역별 파일 개수:", len(region_files))
# for r, p in region_files.items():
#     count = assigned_df[assigned_df["region"] == r].shape[0]
#     print(f"{r}: {count}건 -> {p}")

전체 할당 저장 완료: output_assigned\restareas_assigned_all_20260329_131441.csv
